# Chapter 5: Cross-Validation Exercises
## Question 8 — Simulated data, LOOCV, and model significance

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import LeaveOneOut, cross_val_score

### (a) Generate the simulated data set
n = 100, p = 1. True model: $Y = X - 2X^2 + \varepsilon$, $\varepsilon \sim N(0,1)$

In [ ]:
rng = np.random.default_rng(1)
x = rng.normal(size=100)
y = x - 2 * x**2 + rng.normal(size=100)

print("n =", len(x), " p =", 1)
df = pd.DataFrame({'x': x, 'y': y})
df.head()

### (b) Scatterplot of X vs Y

In [ ]:
plt.figure(figsize=(6,4.5))
plt.scatter(x, y, alpha=0.7, edgecolor='k')
plt.xlabel('X')
plt.ylabel('Y')
plt.title('Scatterplot of Y vs X (simulated data)')
plt.tight_layout()
plt.show()

**Comment:** X ranges roughly from -2.7 to 2.1, and Y is clearly a curved (concave-down, quadratic) function of X, not linear — it rises then falls, peaking near X ≈ 0. This matches the generating model Y = X - 2X² + ε.

### (c) & (d) LOOCV errors for polynomial degrees 1-4, with two random seeds

In [ ]:
def loocv_errors(x, y, degree, seed=None):
    if seed is not None:
        np.random.seed(seed)  # set for the exercise; OLS + LOOCV is deterministic regardless
    X = np.column_stack([x**d for d in range(1, degree+1)])
    loo = LeaveOneOut()
    model = LinearRegression()
    mse = -cross_val_score(model, X, y, cv=loo,
                            scoring='neg_mean_squared_error').mean()
    return mse

results = {}
for seed in [1, 2]:
    print(f"Seed {seed}:")
    results[seed] = {}
    for d in range(1, 5):
        err = loocv_errors(x, y, d, seed=seed)
        results[seed][d] = err
        print(f"  degree {d}: {err:.4f}")

**(d) Same seed, different results?** No — identical across seeds. LOOCV leaves out exactly one specific observation per fold with no random fold assignment involved (unlike k-fold with k < n), and OLS fitting is deterministic. So the seed has no effect here.

### (e) Smallest LOOCV error
The **quadratic (degree 2)** model has the smallest LOOCV error. This is expected since the true relationship is quadratic: Y = X - 2X² + ε. The linear model underfits; cubic/quartic terms add no real signal and slightly increase error.

### (f) Statistical significance of coefficients

In [ ]:
models = {}
for d in range(1, 5):
    X = np.column_stack([x**k for k in range(1, d+1)])
    X = sm.add_constant(X)
    model = sm.OLS(y, X).fit()
    models[d] = model
    print(f"\n{'='*70}\nDEGREE {d} MODEL\n{'='*70}")
    print(model.summary())

**Discussion:** R² jumps from 0.318 (linear) to 0.887 (quadratic), the biggest gain by far coming from adding X² — matching LOOCV's big drop in error. R² barely moves from degree 2 → 3 → 4, and the cubic term (X³) is never statistically significant (p = 0.287, p = 0.642). X⁴ is marginally significant (p = 0.023) in the degree-4 model, but this is best read as overfitting noise rather than genuine signal. These significance results agree well with the LOOCV conclusions: only X and X² are consistently important, matching the true generating model and the fact that degree 2 minimizes LOOCV error.